In [4]:
!pip install -q transformers datasets evaluate accelerate torch scikit-learn huggingface_hub safetensors

In [14]:
# Quick experiment: collapse family variants into one label, see if it helps
import re
def collapse_family(lbl):
    return re.sub(r'(\d*)\.bvh$', '.bvh', lbl)

collapsed_labels_raw = [collapse_family(l) for l in labels_raw]
print(f"Classes before collapse: {len(set(labels_raw))}  |  after: {len(set(collapsed_labels_raw))}")

Classes before collapse: 44  |  after: 33


In [16]:
# ===== Kaggle Notebook: Marin Text-to-Animation Tagger (v7) =====
# Upload this notebook, marin_animation_dataset.jsonl, and marin_gesture_chunks.json to Kaggle.
# Turn on GPU (T4 is fine).
#
# Changes from v6:
#   v6 fixed backbone loading (gamma/beta rename + bert. prefix restore) and
#   fixed too-few-training-steps (batch 32->8, epochs 8->25). That got
#   accuracy from ~0.13 to ~0.35, but the train/val loss gap kept widening
#   (train loss 7.46->3.69, val loss only 7.33->5.32, plateauing ~epoch 20)
#   -- classic overfitting on a small dataset. This version adds:
#     1. Higher dropout (0.3) on the backbone to fight overfitting.
#     2. Stronger weight decay (0.1) and label smoothing (0.1).
#     3. Cosine LR schedule instead of linear.
#     4. More epochs (60) but load_best_model_at_end + save_total_limit
#        mean it's safe -- it ships whichever checkpoint had the best
#        eval f1_weighted, not necessarily the last one.
#     5. Explicit print of the full training result + final eval, since the
#        Kaggle progress-bar widget sometimes doesn't survive copy/paste.

#!pip install -q transformers datasets evaluate accelerate torch scikit-learn huggingface_hub safetensors

import json
import torch
from datasets import Dataset
from huggingface_hub import hf_hub_download
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)

# ─── CONFIG ───────────────────────────────────────────────────────────────────
KAGGLE_BASE  = "/kaggle/input/datasets/bayazidhs/marin-vibe"
ANIM_JSONL   = f"{KAGGLE_BASE}/marin_animation_dataset.jsonl"
GESTURE_JSON = f"{KAGGLE_BASE}/marin_gesture_chunks.json"

MODEL_NAME = "microsoft/MiniLM-L12-H384-uncased"  # ~33M params
MAX_LENGTH = 64

# Match the same speaking-rate assumption used to build marin_gesture_chunks.json,
# so re-chunked augmentation data lines up with the existing chunk style.
SPEAKING_RATE_CHARS_PER_SEC = 15.0

# ─── 1. LOAD GESTURE CHUNKS DATASET (primary source, already correctly chunked) ──
print("Loading gesture chunks dataset...")
with open(GESTURE_JSON, "r", encoding="utf-8") as f:
    gesture_data = json.load(f)

texts      = list(gesture_data["texts"])
labels_raw = list(gesture_data["labels"])
all_labels = gesture_data["all_labels"]

label2id = {lbl: i for i, lbl in enumerate(all_labels)}
id2label = {i: lbl for i, lbl in enumerate(all_labels)}
num_labels = len(all_labels)
print(f"  {len(texts)} samples | {num_labels} animation classes")

# ─── 2. AUGMENT WITH RE-CHUNKED FULL ANIMATION JSONL ─────────────────────────
def snap_to_word_boundary(pos, text):
    """Nudge a raw character offset forward to the next whitespace so we never
    slice a chunk boundary in the middle of a word."""
    if pos <= 0 or pos >= len(text):
        return pos
    while pos < len(text) and text[pos] not in " \t\n":
        pos += 1
    return pos

def build_chunks(item):
    text = item.get("text", "")
    seqs = sorted(item.get("sequence", []), key=lambda s: s["time_seconds"])
    if not text or not seqs:
        return []
    offsets = [(min(s["time_seconds"] * SPEAKING_RATE_CHARS_PER_SEC, len(text)), s["animation"])
               for s in seqs]
    chunks = []
    for i, (start, anim) in enumerate(offsets):
        end = offsets[i + 1][0] if i + 1 < len(offsets) else len(text)
        start_i = snap_to_word_boundary(int(start), text)
        end_i = snap_to_word_boundary(int(end), text)
        chunk_text = text[start_i:end_i].strip()
        # Drop degenerate leftover fragments (punctuation-only, 1-2 word scraps)
        if chunk_text and len(chunk_text.split()) >= 3 and anim in label2id:
            chunks.append((chunk_text, anim))
    return chunks

print("Augmenting with re-chunked marin_animation_dataset.jsonl ...")
extra_texts, extra_labels = [], []
try:
    with open(ANIM_JSONL, "r", encoding="utf-8") as f:
        for line in f:
            item = json.loads(line)
            for chunk_text, anim in build_chunks(item):
                extra_texts.append(chunk_text)
                extra_labels.append(anim)
    texts      += extra_texts
    labels_raw += extra_labels
    print(f"  +{len(extra_texts)} re-chunked augmented examples -> total {len(texts)}")
except FileNotFoundError:
    print(f"  WARNING: {ANIM_JSONL} not found, skipping augmentation.")

# Drop exact-duplicate (text, label) pairs that can arise from overlapping sources
seen = set()
dedup_texts, dedup_labels = [], []
for t, l in zip(texts, labels_raw):
    key = (t, l)
    if key not in seen:
        seen.add(key)
        dedup_texts.append(t)
        dedup_labels.append(l)
removed = len(texts) - len(dedup_texts)
if removed:
    print(f"  Removed {removed} exact duplicate (text,label) pairs")
texts, labels_raw = dedup_texts, dedup_labels

print("\n--- Sample re-chunked pairs (spot-check boundaries) ---")
for t, l in list(zip(extra_texts, extra_labels))[:8]:
    print(f"  [{l}] {t!r}")
print()

label_ids = [label2id[lbl] for lbl in labels_raw]

# ─── 3. TOKENIZER & MODEL (gamma/beta fix + base_model_prefix fix + dropout) ─
def load_fixed_state_dict(model_name, base_prefix):
    """Download the checkpoint's raw weights and:
      1. rename any legacy TF-style LayerNorm keys (.gamma/.beta) to
         HF-standard (.weight/.bias)
      2. add back the base-model prefix (e.g. "bert.") that HF normally
         inserts automatically when loading a base checkpoint into a model
         with a task head -- lost when we load the state dict manually.
    """
    try:
        ckpt_path = hf_hub_download(model_name, filename="pytorch_model.bin")
        raw_state_dict = torch.load(ckpt_path, map_location="cpu")
    except Exception:
        from safetensors.torch import load_file
        ckpt_path = hf_hub_download(model_name, filename="model.safetensors")
        raw_state_dict = load_file(ckpt_path)

    fixed = {}
    renamed = 0
    reprefixed = 0
    for k, v in raw_state_dict.items():
        new_k = k.replace(".gamma", ".weight").replace(".beta", ".bias")
        if new_k != k:
            renamed += 1
        if not new_k.startswith(base_prefix + "."):
            new_k = f"{base_prefix}.{new_k}"
            reprefixed += 1
        fixed[new_k] = v
    print(f"  Renamed {renamed} legacy gamma/beta keys -> weight/bias")
    print(f"  Added '{base_prefix}.' prefix to {reprefixed} keys")
    return fixed

print(f"Loading tokenizer and model: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config = AutoConfig.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    hidden_dropout_prob=0.3,            # was default ~0.1 -- more dropout for a small dataset
    attention_probs_dropout_prob=0.3,   # same
)
model = AutoModelForSequenceClassification.from_config(config)

fixed_state_dict = load_fixed_state_dict(MODEL_NAME, model.base_model_prefix)

load_result = model.load_state_dict(fixed_state_dict, strict=False)
print("  Missing keys:", load_result.missing_keys)
print("  Unexpected keys:", load_result.unexpected_keys)
# Expect ONLY classifier.weight / classifier.bias in missing_keys, and
# unexpected_keys empty or containing only a harmless buffer like
# "bert.embeddings.position_ids". If any encoder/LayerNorm keys still show
# up in either list, stop and paste this output before training.

# ─── 4. PREPARE DATASET ──────────────────────────────────────────────────────
print("Tokenizing dataset...")
def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH, padding="max_length")

raw_ds = Dataset.from_dict({"text": texts, "label": label_ids})
raw_ds = raw_ds.train_test_split(test_size=0.1, seed=42)

train_ds = raw_ds["train"].map(tokenize_fn, batched=True)
eval_ds  = raw_ds["test"].map(tokenize_fn, batched=True)

train_ds = train_ds.remove_columns(["text"])
eval_ds  = eval_ds.remove_columns(["text"])
train_ds.set_format("torch")
eval_ds.set_format("torch")

print(f"  Train: {len(train_ds)} | Eval: {len(eval_ds)}")

# ─── 5. METRICS ──────────────────────────────────────────────────────────────
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1  = f1_score(labels, preds, average="weighted", zero_division=0)
    return {"accuracy": acc, "f1_weighted": f1}

# ─── 6. TRAINING (stronger regularization + explicit summary printout) ──────
print("\nStarting training...")
training_args = TrainingArguments(
    output_dir="./animation_model_output",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=32,
    num_train_epochs=60,                # load_best_model_at_end ships the best
                                         # checkpoint, not necessarily the last one
    weight_decay=0.1,                   # was 0.01 -- stronger L2 penalty
    label_smoothing_factor=0.1,         # discourages overconfident wrong predictions
    lr_scheduler_type="cosine",         # was linear default -- smoother decay
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,                 # avoid piling up 60 checkpoints on disk
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    logging_steps=20,
    warmup_ratio=0.1,
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    compute_metrics=compute_metrics,
)

train_result = trainer.train()

print("\n=== Final training summary ===")
print(train_result)

final_eval = trainer.evaluate()
print("\n=== Final eval metrics ===")
print(final_eval)

# ─── 7. SAVE & EXPORT (same format director_engine.py already expects) ──────
print("\nSaving model...")
model.save_pretrained("marin_animation_model")
tokenizer.save_pretrained("marin_animation_model")

with open("marin_animation_model/label_map.json", "w") as f:
    json.dump({"id2label": id2label, "label2id": label2id, "all_labels": all_labels}, f, indent=2)

torch.save(model.state_dict(), "marin_animation_weights.pt")

print("Done! Download marin_animation_model/ (HF format) + marin_animation_weights.pt")
print(f"Model predicts one of {num_labels} animations per text chunk.")
print(f"Backbone: {MODEL_NAME}")

Loading gesture chunks dataset...
  1168 samples | 44 animation classes
Augmenting with re-chunked marin_animation_dataset.jsonl ...
  +945 re-chunked augmented examples -> total 2113
  Removed 140 exact duplicate (text,label) pairs

--- Sample re-chunked pairs (spot-check boundaries) ---
  [neutral_idle2.bvh] "I've reviewed your last module, Limon. Your logic is flawless"
  [caring1.bvh] "and your implementation is precise. It seems you're finally"
  [gratitude.bvh] 'evolving into something useful. Keep this momentum, and you might actually survive my curriculum.'
  [surprise2.bvh] "Look at those benchmarks. You didn't just meet the target; you crushed it. I'm"
  [action_pat.bvh] "almost impressed, you clever little sod. Don't let it go"
  [neutral_idle2.bvh] 'to your head—complacency is the first step toward the grave.'
  [neutral_idle2.bvh] "Clean code. Optimized memory. Disciplined execution. This is exactly why I haven't scrapped"
  [gratitude.bvh] "you yet, Limon. You're starting

Map:   0%|          | 0/1775 [00:00<?, ? examples/s]

Map:   0%|          | 0/198 [00:00<?, ? examples/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


  Train: 1775 | Eval: 198

Starting training...


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


=== Final training summary ===
TrainOutput(global_step=6660, training_loss=6.164776259642822, metrics={'train_runtime': 586.9351, 'train_samples_per_second': 181.451, 'train_steps_per_second': 11.347, 'total_flos': 877592007936000.0, 'train_loss': 6.164776259642822, 'epoch': 60.0})



=== Final eval metrics ===
{'eval_loss': 6.530354976654053, 'eval_accuracy': 0.1919191919191919, 'eval_f1_weighted': 0.1335414771335848, 'eval_runtime': 0.1749, 'eval_samples_per_second': 1132.239, 'eval_steps_per_second': 22.874, 'epoch': 60.0}

Saving model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Done! Download marin_animation_model/ (HF format) + marin_animation_weights.pt
Model predicts one of 44 animations per text chunk.
Backbone: microsoft/MiniLM-L12-H384-uncased


In [18]:
from collections import Counter

print("=== Label distribution (pre-dedup, all sources combined) ===")
label_counts = Counter(labels_raw)
print(f"Total classes with data: {len(label_counts)} / {num_labels} defined")
print(f"Classes with < 5 examples: {sum(1 for c in label_counts.values() if c < 5)}")
print(f"Classes with < 10 examples: {sum(1 for c in label_counts.values() if c < 10)}")
print("\nBottom 10 rarest classes:")
for lbl, cnt in sorted(label_counts.items(), key=lambda x: x[1])[:10]:
    print(f"  {lbl}: {cnt}")
print("\nTop 10 most common classes:")
for lbl, cnt in sorted(label_counts.items(), key=lambda x: -x[1])[:10]:
    print(f"  {lbl}: {cnt}")

print("\n=== Chunk length stats (word count) ===")
lengths = [len(t.split()) for t in texts]
import numpy as np
print(f"  mean: {np.mean(lengths):.1f}  median: {np.median(lengths):.1f}  min: {min(lengths)}  max: {max(lengths)}")
print(f"  chunks with <= 3 words: {sum(1 for l in lengths if l <= 3)} / {len(lengths)}")

=== Label distribution (pre-dedup, all sources combined) ===
Total classes with data: 44 / 44 defined
Classes with < 5 examples: 0
Classes with < 10 examples: 3

Bottom 10 rarest classes:
  sit_idle4.bvh: 7
  kneel_idle2.bvh: 7
  hitarea_groin.bvh: 9
  action_laydown.bvh: 10
  laying_idle2.bvh: 10
  hitarea_foot.bvh: 10
  desire.bvh: 10
  exercise_jogging.bvh: 13
  kneel_idle.bvh: 14
  laying_idle.bvh: 14

Top 10 most common classes:
  surprise2.bvh: 139
  disaproval1.bvh: 112
  neutral3.bvh: 105
  neutral_idle2.bvh: 102
  pride.bvh: 93
  amusement2.bvh: 92
  relief1.bvh: 87
  neutral2.bvh: 79
  confusion.bvh: 67
  gratitude.bvh: 66

=== Chunk length stats (word count) ===
  mean: 10.2  median: 10.0  min: 1  max: 22
  chunks with <= 3 words: 62 / 1973


In [19]:
from collections import defaultdict

text_to_labels = defaultdict(set)
for t, l in zip(texts, labels_raw):
    text_to_labels[t.strip().lower()].add(l)

conflicts = {t: lbls for t, lbls in text_to_labels.items() if len(lbls) > 1}
print(f"Exact-text conflicts: {len(conflicts)} texts mapped to multiple labels")
for t, lbls in list(conflicts.items())[:10]:
    print(f"  {lbls}: {t!r}")

# Check for "variant family" collapse potential -- same animation root, different suffix number
import re
families = defaultdict(set)
for lbl in all_labels:
    root = re.sub(r'\d*\.bvh$', '', lbl)
    families[root].add(lbl)
multi_variant = {r: v for r, v in families.items() if len(v) > 1}
print(f"\nAnimation 'families' with multiple numbered variants: {len(multi_variant)}")
for root, variants in multi_variant.items():
    print(f"  {root}: {sorted(variants)}")

Exact-text conflicts: 10 texts mapped to multiple labels
  {'optimism.bvh', 'hitarea_groin.bvh', 'neutral.bvh', 'action_laydown.bvh'}: '.'
  {'disaproval1.bvh', 'curiosity3.bvh', 'pride2.bvh'}: "still staring at that bug like it's going to solve itself? y"
  {'curiosity3.bvh', 'surprise2.bvh'}: "you actually finished the module on time? i'm"
  {'neutral.bvh', 'pride.bvh'}: 'or get out of my sight.'
  {'disappointment.bvh', 'anger2.bvh'}: "ou're a bloody plonker, limon. fix the logic or start pushin"
  {'gratitude.bvh', 'pride.bvh'}: "almost impressed. don't let it go to your head, you"
  {'action_pat.bvh', 'gratitude.bvh'}: 'proud of you.'
  {'exercise_crunches.bvh', 'pride.bvh'}: 'your schedule becomes a living nightmare.'
  {'curiosity3.bvh', 'disaproval1.bvh'}: "still staring at that bug like it's going to solve itself? you're"
  {'disappointment.bvh', 'anger2.bvh'}: 'a bloody plonker, limon. fix the logic or start pushing'

Animation 'families' with multiple numbered variants: 8
  

In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

X_train, X_test, y_train, y_test = train_test_split(texts, label_ids, test_size=0.1, random_state=42)

vec = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_train_v = vec.fit_transform(X_train)
X_test_v  = vec.transform(X_test)

clf = LogisticRegression(max_iter=1000, class_weight="balanced")
clf.fit(X_train_v, y_train)
preds = clf.predict(X_test_v)

print("TF-IDF + LogisticRegression baseline:")
print(f"  accuracy: {accuracy_score(y_test, preds):.3f}")
print(f"  f1_weighted: {f1_score(y_test, preds, average='weighted', zero_division=0):.3f}")

TF-IDF + LogisticRegression baseline:
  accuracy: 0.505
  f1_weighted: 0.499


In [21]:
print("\n--- Sample re-chunked pairs (spot-check boundaries) ---")
for t, l in list(zip(extra_texts, extra_labels))[:8]:
    print(f"  [{l}] {t!r}")


--- Sample re-chunked pairs (spot-check boundaries) ---
  [neutral_idle2.bvh] "I've reviewed your last module, Limon. Your logic is flawless"
  [caring1.bvh] "and your implementation is precise. It seems you're finally"
  [gratitude.bvh] 'evolving into something useful. Keep this momentum, and you might actually survive my curriculum.'
  [surprise2.bvh] "Look at those benchmarks. You didn't just meet the target; you crushed it. I'm"
  [action_pat.bvh] "almost impressed, you clever little sod. Don't let it go"
  [neutral_idle2.bvh] 'to your head—complacency is the first step toward the grave.'
  [neutral_idle2.bvh] "Clean code. Optimized memory. Disciplined execution. This is exactly why I haven't scrapped"
  [gratitude.bvh] "you yet, Limon. You're starting to think like a machine. I'm"


In [ ]:
# ─── SAVE SKLEARN MODEL (run this, then download the .pkl) ──────────────────
import pickle

save_obj = {
    "vectorizer": vec,
    "classifier": clf,
    "id2label":   id2label,
    "label2id":   label2id,
    "all_labels": all_labels,
}
with open("marin_sklearn_gesture.pkl", "wb") as f:
    pickle.dump(save_obj, f)

print("Done! File size:", round(__import__('os').path.getsize('marin_sklearn_gesture.pkl') / 1024, 1), "KB")
print("Download: marin_sklearn_gesture.pkl → place in /home/sword/Documents/marin/")
